In [1]:
import numpy as np

## Inputs as one row per bond:

- N = number of bonds ever issued
- issue_period[i] = which calendar period (index into a global semiannual timeline) bond i was issued
- T[i] = original maturity length in years → maturity_period[i] = issue_period[i] + 2*T[i]
- C[i] = coupon rate set at bond i's own auction (this is why it has to be a vector: every bond's coupon is a frozen snapshot of a different point in history, exactly what we established a few turns back)
- F[i] = face value issued for bond i

In [2]:
issue_period=np.array([0,2,4])
T=np.array([2,3,1])
C=np.array([0.02,0.04,0.035])
F=np.full(len(T),100)

N=len(issue_period)

periods_to_maturity=2*T
maturity_period=periods_to_maturity+issue_period

## Problem 1: Build the `Active` mask

**Statement:**
Given `issue_period` and `maturity_period` (one entry per bond) and `periods` (the global calendar grid, one entry per period), build a boolean matrix `Active` of shape `(N, M)` where:

```
Active[i][t] = True   if t > issue_period[i]  AND  t <= maturity_period[i]
Active[i][t] = False  otherwise
```

In words: bond `i` has a live payment obligation at period `t` if that period falls strictly after its issue date and at or before its maturity date.

**Constraint:** no explicit `for` loops over bonds or periods. Use broadcasting (`periods[None, :]` against `issue_period[:, None]` and `maturity_period[:, None]`) so the whole `(N, M)` grid is produced by vectorized comparisons, and combine the two conditions with `&`.

**Input** (already defined above):
```
issue_period      = [0, 2, 4]
maturity_period   = [4, 8, 6]
periods            = [0, 1, 2, 3, 4, 5, 6, 7, 8]   (M = 9)
```

**Expected Output** (shape `(3, 9)`, dtype bool):
```
[[False,  True,  True,  True,  True, False, False, False, False],
 [False, False, False,  True,  True,  True,  True,  True,  True],
 [False, False, False, False, False,  True,  True, False, False]]
```

Row 0 (bond 0, issued t=0, matures t=4): live at t = 1,2,3,4.
Row 1 (bond 1, issued t=2, matures t=8): live at t = 3,4,5,6,7,8.
Row 2 (bond 2, issued t=4, matures t=6): live at t = 5,6.

**Hint on your current cell below:** it always assigns `True` regardless of `t`, that's the placeholder to replace, and it's also still a nested loop, which the constraint above rules out.


In [3]:
M=maturity_period.max()+1
periods=np.arange(M)

Active=(periods[None,:]>issue_period[:,None]) & (periods[None,:]<=maturity_period[:,None])
Active

array([[False,  True,  True,  True,  True, False, False, False, False],
       [False, False, False,  True,  True,  True,  True,  True,  True],
       [False, False, False, False, False,  True,  True, False, False]])

## Problem 2: Build `CouponFlow`, `PrincipalFlow`, `Outstanding`

**Objective:** Once `Active` is correct, use it (plus `C`, `F`, `issue_period`, `maturity_period`) to build three more `(N, M)` arrays that together describe every dollar moving in or out of each bond at each period.

**Explanation:**
- `CouponFlow[i,t]` fires on every period bond `i` is `Active`, since your Problem 1 mask already encodes "has a live payment obligation," including the maturity period itself (the final coupon is paid alongside principal). The semiannual amount is `C[i]/2 * F[i]`, frozen at issuance for the bond's whole life.
- `PrincipalFlow[i,t]` fires exactly once, at `maturity_period[i]`, paying back `F[i]`.
- `Outstanding[i,t]` answers a different question than `Active`: not "does cash move today" but "is this bond currently sitting in the debt stock." A bond becomes debt the moment it's issued (`t >= issue_period[i]`, note `>=` not `>`), and drops out of the stock once it's repaid (`t < maturity_period[i]`, note `<` not `<=`, since it's retired exactly at maturity, not still outstanding then). `Active` and `Outstanding` look almost identical but use opposite-direction boundary conditions on both ends; mixing them up is the easiest bug to introduce here.

**Constraint:** same as Problem 1, no `for` loops. `CouponFlow` and `Outstanding` are each a broadcasted boolean mask times a per-bond scalar (broadcast `(N,)` against `(N, M)` via `[:, None]`); `PrincipalFlow` is an equality mask (`periods[None,:] == maturity_period[:,None]`) times `F`.

**Input:** reuse `issue_period`, `maturity_period`, `C`, `F` from above, plus your corrected `Active` from Problem 1.

**Expected Output** (each shape `(3, 9)`):

`CouponFlow`, nonzero value is `C[i]/2 * F[i]`: bond 0 = 1.0, bond 1 = 2.0, bond 2 = 1.75
```
[[0.  , 1.  , 1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  ],
 [0.  , 0.  , 0.  , 2.  , 2.  , 2.  , 2.  , 2.  , 2.  ],
 [0.  , 0.  , 0.  , 0.  , 0.  , 1.75, 1.75, 0.  , 0.  ]]
```

`PrincipalFlow`, 100 exactly at each bond's own `maturity_period` (4, 8, 6):
```
[[0., 0., 0., 0., 100., 0., 0., 0., 0.],
 [0., 0., 0., 0., 0., 0., 0., 0., 100.],
 [0., 0., 0., 0., 0., 0., 100., 0., 0.]]
```

`Outstanding`, 100 for `issue_period[i] <= t < maturity_period[i]`:
```
[[100., 100., 100., 100., 0., 0., 0., 0., 0.],
 [0., 0., 100., 100., 100., 100., 100., 100., 0.],
 [0., 0., 0., 0., 100., 100., 0., 0., 0.]]
```

In [4]:
CouponFlow = Active*((C/2)*F)[:,None]
PrincipalFlow = (periods[None,:] == maturity_period[:,None])*F[:,None].astype(float)
Outstanding = ((periods[None,:]>=issue_period[:,None]) & (periods[None,:]<maturity_period[:,None]))*F[:,None].astype(float)

CouponFlow,PrincipalFlow,Outstanding

(array([[0.  , 1.  , 1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 2.  , 2.  , 2.  , 2.  , 2.  , 2.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 1.75, 1.75, 0.  , 0.  ]]),
 array([[  0.,   0.,   0.,   0., 100.,   0.,   0.,   0.,   0.],
        [  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0., 100.],
        [  0.,   0.,   0.,   0.,   0.,   0., 100.,   0.,   0.]]),
 array([[100., 100., 100., 100.,   0.,   0.,   0.,   0.,   0.],
        [  0.,   0., 100., 100., 100., 100., 100., 100.,   0.],
        [  0.,   0.,   0.,   0., 100., 100.,   0.,   0.,   0.]]))

## Problem 3: Column sums, aggregate the debt stock and cash flows

**Objective:** collapse the bond dimension (`axis=0`) on each of the three `(N, M)` arrays from Problem 2 to get the aggregate time series a real debt-management desk would actually watch, and hand-check every value against the toy dataset.

**Explanation:** each column `t` sums a story across all three bonds at once: some are outstanding, some just matured, most are silent. `Outstanding[:, t].sum()` is the total debt stock at `t`. `CouponFlow[:, t].sum()` is the interest bill at `t` (excludes principal). Adding `PrincipalFlow[:, t].sum()` on top gives total debt service, what actually gets paid out that period including redemptions. This step also doubles as your correctness check on Problems 1 and 2: a wrong boundary condition in `Active` or `Outstanding` usually still looks plausible cell by cell, but produces an aggregate series that is obviously wrong once you sum it and compare to a number you can verify by eye.

**Constraint:** one `.sum(axis=0)` call per array, no loops.

**Input:** your `CouponFlow`, `PrincipalFlow`, `Outstanding` from Problem 2.

**Expected Output** (each length 9, indexed `t = 0..8`):
```
Outstanding total    (debt stock):        [100, 100, 200, 200, 200, 200, 100, 100,   0]
CouponFlow total     (interest only):     [  0,   1,   1,   3,   3, 3.75, 3.75,   2,   2]
Total debt service   (interest + principal, spikes at t=4,6,8 when a bond matures):
                                           [  0,   1,   1,   3, 103, 3.75, 103.75,  2, 102]
```

Sanity checks worth doing by hand: at `t=4` bond 0 matures, so total debt service jumps by its face value (100) on top of the coupon bill that period. Debt stock hits 0 at `t=8` because all three bonds have matured by then (bond 1 is the last, and it drops out of `Outstanding` at `t=8` itself, matching the `<` boundary from Problem 2, not `<=`).

In [5]:
print("Outstanding total\t(debt stock):\t",Outstanding.sum(axis=0))
print("Coupon Flow total\t(interest only):\t",CouponFlow.sum(axis=0))
print("Total debt service\t(interest + principal, spikes at t=4,6,8 when a bond matures):\t",PrincipalFlow.sum(axis=0)+CouponFlow.sum(axis=0))


Outstanding total	(debt stock):	 [100. 100. 200. 200. 200. 200. 100. 100.   0.]
Coupon Flow total	(interest only):	 [0.   1.   1.   3.   3.   3.75 3.75 2.   2.  ]
Total debt service	(interest + principal, spikes at t=4,6,8 when a bond matures):	 [  0.     1.     1.     3.   103.     3.75 103.75   2.   102.  ]


## Problem 4: Build the discount-factor scenario matrix

**Objective:** stress-test the debt stock against a handful of simulated/shocked rate paths. Build `DF_scenarios`, shape `(K, M)`, one discount-factor curve per scenario.

**Explanation:** each scenario `k` is a flat annualized rate (a simple stand-in for "the whole curve shifts to this level"), compounded semiannually since periods are semiannual steps. The discount factor for period `t` under scenario `k` is `DF[k,t] = 1 / (1 + rate[k]/2)**t`, so every extra period compounds the discount one more time. `t=0` (today) always discounts to exactly `1.0`, a good self-check.

**Constraint:** no loops. Broadcast `scenario_rates[:,None]` (shape `(K,1)`) against `periods[None,:]` (shape `(1,M)`) so the whole `(K,M)` grid comes out of one vectorized expression.

**Input:**
```
scenario_rates = [0.02, 0.04, 0.06]   # annual rate, "down"/"base"/"up" (K=3)
periods        = [0..8]               # from Problem 1 (M=9)
```

**Expected Output** (shape `(3, 9)`, rounded to 4 decimals; `t=0` column is `1.0` in every row):
```
down (1% semiannual): [1.0, 0.9901, 0.9803, 0.9706, 0.961 , 0.9515, 0.942 , 0.9327, 0.9235]
base (2% semiannual): [1.0, 0.9804, 0.9611, 0.9423, 0.9238, 0.9057, 0.888 , 0.8706, 0.8535]
up   (3% semiannual): [1.0, 0.9709, 0.9426, 0.9151, 0.8885, 0.8626, 0.8375, 0.8131, 0.7894]
```

In [6]:
scenario_rates=np.array([0.02,0.04,0.06])  # down/base/up annual scenario rates
K=len(scenario_rates)

DF_scenarios=1/(1+scenario_rates[:,None]/2)**periods[None,:]
DF_scenarios

array([[1.        , 0.99009901, 0.98029605, 0.97059015, 0.96098034,
        0.95146569, 0.94204524, 0.93271805, 0.92348322],
       [1.        , 0.98039216, 0.96116878, 0.94232233, 0.92384543,
        0.90573081, 0.88797138, 0.87056018, 0.85349037],
       [1.        , 0.97087379, 0.94259591, 0.91514166, 0.88848705,
        0.86260878, 0.83748426, 0.81309151, 0.78940923]])

## Problem 5: Sensitivity profile, discount total debt service under every scenario

**Objective:** collapse everything built so far into one number per scenario: the present value of total future debt service under each rate shock, the sensitivity profile the README describes.

**Explanation:** `debt_service_total` from Problem 3 already has the bond dimension collapsed away, it is exactly the `TotalFutureCF` vector the design doc talks about. Pricing it under every scenario at once is a single matmul: `DF_scenarios (K,M) @ TotalFutureCF (M,)` gives a length-`K` vector, `pv[k] = sum over t of DF_scenarios[k,t] * TotalFutureCF[t]`, the present value under scenario `k`. No explicit loop over scenarios or periods, the matmul does both sums in one shot.

**Constraint:** one `@` (or `np.matmul`), no loops.

**Input:** `DF_scenarios` from Problem 4, `debt_service_total` from Problem 3.

**Expected Output** (length 3, one PV per scenario):
```
[down_PV, base_PV, up_PV] ~ [301.2, 284.2, 268.4]
```
The direction is the real check: lower rates give a higher present value, higher rates give a lower one, the same inverse rate/price relationship that drives duration risk on a real book.

In [7]:
TotalFutureCF=CouponFlow.sum(axis=0)+PrincipalFlow.sum(axis=0)

pv_scenarios=DF_scenarios@TotalFutureCF
pv_scenarios

array([301.22905527, 284.24526648, 268.4427601 ])